#Load Dataset & Understand it

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/tweet_product_company.csv")  # change this path

# View first 5 rows
df

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
## Drop unnecessary column
df = df.drop("emotion_in_tweet_is_directed_at", axis=1)

For better readability, lets rename column names to 'text' and 'sentiment'.


In [ ]:
df.columns = ["text", "sentiment"]

In [ ]:
df.head()

In [ ]:
#check for null values again
df.isnull().sum()

In [ ]:
round(df.isnull().mean()*100,2)

In [ ]:
#drop the row
df = df.dropna()

In [ ]:
df.isnull().sum()

#Text Preprocessing

In [ ]:
#import libraries

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

In [ ]:
#Create Cleaning Function

In [ ]:
def clean_text(text):
    text = text.lower()  # lowercase

    # remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|@\w+|#\w+', '', text)

    # remove special characters & numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]

    return " ".join(words)

In [ ]:
#Apply Cleaning

In [ ]:
df['clean_text'] = df['text'].apply(clean_text)

In [ ]:
#Verify Output

In [ ]:
df[['text', 'clean_text']].head()

In [ ]:
#Performed text preprocessing including lowercasing, removal of URLs, mentions, special characters, and stopwords to clean the text data.

#Encoding

In [ ]:
#use label encoder
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()

df['label'] = le.fit_transform(df['sentiment'])

In [ ]:
print(df[['sentiment', 'label']].head())

In [ ]:
print(le.classes_)

In [ ]:
#Simplify Labels

In [ ]:
df['sentiment'] = df['sentiment'].replace({
    "Positive emotion": "positive",
    "Negative emotion": "negative",
    "No emotion toward brand or product": "neutral",
    "I can't tell": "no_idea"
})

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])

In [ ]:
print(le.classes_)

#Tokenization & Padding

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
tokenizer = Tokenizer(num_words=5000)

In [ ]:
tokenizer.fit_on_texts(df['clean_text'])

In [ ]:
sequences = tokenizer.texts_to_sequences(df['clean_text'])

In [ ]:
X = pad_sequences(sequences, maxlen=100)

In [ ]:
y = df['label']

In [ ]:
print(X.shape)
print(y.shape)

#Train-Test Split + Model Building (RNN / LSTM)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)

In [ ]:
#Build LSTM Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=5000, output_dim=128, input_length=100),
    LSTM(64),
    Dense(4, activation='softmax')  # 4 classes
])

In [ ]:
#Compile Model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
#Train Model
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

#Prediction and Evaluation

In [ ]:
#Evaluate model
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc)

In [ ]:
# Test multiple sentences
new_texts = [
    "I love Apple products",
    "I hate this Apple product",
    "This phone is okay",
    "I can't say anything about this brand"
]

# Preprocess
cleaned = [clean_text(t) for t in new_texts]

#Tokenize
seq = tokenizer.texts_to_sequences(cleaned)

# Pad
padded = pad_sequences(seq, maxlen=100)

# Predict
pred = model.predict(padded)

# Get class index
pred_classes = np.argmax(pred, axis=1)

# Convert to actual labels
pred_labels = le.inverse_transform(pred_classes)

#Display results
for text, label in zip(new_texts, pred_labels):
    print(f"Text: {text}")
    print(f"Predicted Sentiment: {label}")
    print("-" * 50)

In [ ]:
print(le.classes_)

#BERT Model for Sentiment Classification

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

In [ ]:
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

In [ ]:
texts = [
    "I love Apple products",
    "I hate this Apple product",
    "This phone is okay",
    "I can't say anything about this brand"
]

results = classifier(texts)

for text, res in zip(texts, results):
    print(f"Text: {text}")
    print(f"Predicted: {res['label']} (Confidence: {res['score']:.2f})")
    print("-" * 50)



*   BERT provides highly accurate predictions for positive and negative sentiments.
*   It performs better than LSTM in understanding context.



*   Limitation:It cannot directly classify into 4 classes (positive, negative, neutral, no idea).





